In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
import pandas as pd

2025-09-23 16:42:26.198923: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-23 16:42:26.199409: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-23 16:42:26.256612: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-23 16:42:27.422832: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

In [40]:
x = np.linspace(-10, 10, 1000)
y = 5*x**5 - 4*x**4 + 3*x**3 - 2*x**2 + x - 1
x_train = x[:800]
y_train = y[:800]
x_test = x[800:]
y_test = y[800:]

In [41]:
model = models.Sequential([
    layers.Dense(8, activation='relu', input_dim=1, use_bias=True),
    layers.Dense(8, activation='relu', use_bias=True),
    layers.Dense(8, activation='relu', use_bias=True),
    layers.Dense(1)
])

/home/kirill/code/Python/ML/ML/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [55]:
class WeightChangeCallback(callbacks.Callback):
    def __init__(self):
        super(WeightChangeCallback, self).__init__()
        self.previous_weights = None
        self.weight_diff = dict()

    def on_epoch_end(self, epoch, logs=None):
        current_weights = self.model.get_weights()

        if self.previous_weights is not None and (epoch)%20 == 0:
            for i, (prev, curr) in enumerate(zip(self.previous_weights, current_weights)):
                weight_diff = prev - curr
                self.weight_diff[i] = prev-curr
                print(f'\nEpoch {epoch+1}, слой {i+1}: изменение весов = {weight_diff.T}')
        self.previous_weights = current_weights

weight_change_callback = WeightChangeCallback()

In [56]:
model.compile(optimizer='adam', loss='mse')
model.fit(x_train, y_train, epochs=500, callbacks=[weight_change_callback])

Epoch 1/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 51660824.0000    
Epoch 2/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 49434280.0000 
Epoch 3/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 49429156.0000
Epoch 4/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 50957736.0000 
Epoch 5/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 50025368.0000 
Epoch 6/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 48265288.0000 
Epoch 7/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 49483516.0000 
Epoch 8/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 49318844.0000 
Epoch 9/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 48480560.0000 
Epoch 10/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 47637700.0000 
Epoch 11/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 48519948.0000 
Epoch 12/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 47502032.0000 
Epoch 13/500
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 47963728.0000 
Epoch 14/500
25/25 ━━━━━━━━━━━━━

In [64]:
excel = []
for i in weight_change_callback.weight_diff:
    df = pd.DataFrame(weight_change_callback.weight_diff[i].T)
    excel.append(df)

with pd.ExcelWriter('grad.xlsx', engine='openpyxl') as writer:
    i = 1
    for el in excel:
        el.to_excel(writer, sheet_name=str(i), index=False)
        i+=1

In [51]:
excel = []

for layer in model.layers:
    weights = layer.get_weights()
    print(f'Слой {layer.name}')
    print(f'Weights: {weights[0].T}')
    print(f'Biases: {weights[1]}\n')

    df = pd.DataFrame(weights[0].T)
    # df.to_excel(f'{layer.name}.xlsx', index=False)
    excel.append(df)

    df = pd.DataFrame(weights[1])
    # df.to_excel(f'{layer.name}_biases.xlsx', index=False)
    excel.append(df)

with pd.ExcelWriter('res.xlsx', engine='openpyxl') as writer:
    i = 1
    for el in excel:
        el.to_excel(writer, sheet_name=str(i), index=False)
        i+=1

Слой dense_4
Weights: [[-5.3630476 ]
 [-5.3809443 ]
 [-4.993257  ]
 [ 0.78999615]
 [-5.6750307 ]
 [-5.2850842 ]
 [ 0.73086154]
 [-5.0102243 ]]
Biases: [-1.8080305  -0.01510003 -1.8318381  11.165617   -1.3281461  -1.8704363
 10.523628   -1.3407913 ]

Слой dense_5
Weights: [[-5.2976149e-01  8.1736821e-01 -3.2980248e-01  1.7701292e+01
  -1.2600733e-01 -3.2469437e-01  1.7070276e+01 -6.2474135e-02]
 [-2.4745733e-01  5.5463564e-01 -6.1927354e-01  1.4743796e+01
   1.6370904e-02 -2.9199746e-01  1.5279240e+01  4.5683038e-01]
 [-1.1307310e+01 -1.3870534e+01 -1.0739060e+01  4.3493638e+00
  -1.1834002e+01 -1.0630355e+01  5.0770235e+00 -1.2156618e+01]
 [-2.4290754e-01  5.3636569e-01 -3.9756441e-01  1.6848537e+01
   3.3007669e-01 -2.7863726e-01  1.5926526e+01 -2.5826761e-01]
 [ 4.7047477e+00  4.7735062e+00  4.8541584e+00 -1.6951660e+01
   5.0253696e+00  4.5478654e+00 -1.7340916e+01  4.7330103e+00]
 [ 2.2056819e-01  3.5717648e-01 -5.5903792e-01 -4.3988341e-01
  -3.4292492e-01  4.9557433e-01  4.661589

In [52]:
weight_change_callback.weight_diff

array([0.01478863], dtype=float32)

In [35]:
def relu(x):
    for i in range(len(x)):
        if x[i] < 0:
            x[i] = 0
    return x

In [38]:
x = np.array([2])
for layer in model.layers:
    weights = layer.get_weights()
    print(f'Слой {layer.name}')
    print(f'Weights: {weights[0].T}')
    print(f'Biases: {weights[1]}\n')
    x = (weights[0].T)@x
    print(f'x={x}\n')
    print(f'x+b=')
    x = x + weights[1]
    print(x)
    print(f'\nf(x)={relu(x)}\n\n\n')
    # break

Слой dense
Weights: [[ 1.0693988 ]
 [ 0.4611661 ]
 [-1.2027146 ]
 [-0.714758  ]
 [ 0.88288635]
 [ 0.15453191]
 [-0.01627608]
 [ 0.9784676 ]]
Biases: [-0.54837924 -0.53633547 -1.3162475  -1.8067935  -0.6853101  -0.9432498
 -0.22223677 -0.633159  ]

x=[ 2.13879752  0.92233223 -2.40542912 -1.42951596  1.7657727   0.30906382
 -0.03255215  1.95693517]

x+b=
[ 1.59041828  0.38599676 -3.72167659 -3.23630941  1.08046257 -0.634186
 -0.25478892  1.32377619]

f(x)=[1.59041828 0.38599676 0.         0.         1.08046257 0.
 0.         1.32377619]



Слой dense_1
Weights: [[ 4.91578251e-01  1.83122426e-01  2.48360991e-01  5.67914605e-01
   2.45886698e-01  4.40055847e-01 -1.15651637e-01 -2.20922798e-01]
 [ 1.02284558e-01 -5.56896567e-01  3.88622612e-01 -1.21517634e+00
  -2.35362709e-01  5.76629266e-02 -1.18105739e-01  1.45439744e-01]
 [ 6.41682046e-03  1.69177219e-01  7.02127457e-01  5.32850266e-01
  -2.30000727e-02  3.14787000e-01 -2.14063570e-01 -4.18096721e-01]
 [ 8.37969959e-01  6.41794086e-01  

In [25]:
x = np.array([1, 2, 3, -1, 2, 1])
relu(x)

array([1, 2, 3, 0, 2, 1])